# Silver Layer - Clean and Normalize

**Purpose:** Transform raw bronze data into cleaned, analytics-ready silver tables with proper data types and derived fields.

**Input Tables:**
* `workspace.default.movies_bronze` - Raw movie data

**Output Tables:**
* `workspace.default.movies_silver` - Cleaned movie data with derived fields (release_year, release_decade)

**Transformations:**
* Filter out records with missing critical fields (movie_id, title, release_date)
* Handle NULL values with COALESCE for numeric fields
* Extract release_year from release_date
* Calculate release_decade for trend analysis
* Preserve genre_ids as JSON for downstream exploding

**Execution Order:** Run this notebook after bronze ingestion (01-bronze-ingestion) and before gold aggregations (03-gold-aggregations).

In [0]:
%sql
-- Purpose: Transform bronze data into clean, analytics-ready silver layer
-- Input: workspace.default.movies_bronze
-- Output: workspace.default.movies_silver table

CREATE OR REPLACE TABLE workspace.default.movies_silver
COMMENT 'Silver layer: Cleaned and normalized movie data with derived fields (release_year, release_decade). Updated on-demand after bronze refresh. Source: workspace.default.movies_bronze.'
AS
SELECT
  movie_id,
  title,
  original_title,
  overview,
  release_date,
  COALESCE(popularity, 0.0) AS popularity,
  COALESCE(vote_average, 0.0) AS vote_average,
  COALESCE(vote_count, 0) AS vote_count,
  genre_ids,
  original_language,
  adult,
  poster_path,
  backdrop_path,
  ingestion_timestamp,
  YEAR(TO_DATE(release_date)) AS release_year,
  CAST(FLOOR(YEAR(TO_DATE(release_date)) / 10) * 10 AS INT) AS release_decade
FROM
  workspace.default.movies_bronze
WHERE
  movie_id IS NOT NULL
  AND title IS NOT NULL
  AND release_date IS NOT NULL;

-- Add column comments for business context
ALTER TABLE workspace.default.movies_silver
ALTER COLUMN release_year
COMMENT 'Year extracted from release_date for time-based analysis';

ALTER TABLE workspace.default.movies_silver
ALTER COLUMN release_decade
COMMENT 'Decade of release (e.g., 1990, 2000, 2010) for trend analysis';

ALTER TABLE workspace.default.movies_silver
ALTER COLUMN vote_average
COMMENT 'Average user rating from TMDB users on scale of 0-10';

ALTER TABLE workspace.default.movies_silver
ALTER COLUMN popularity
COMMENT 'TMDB popularity score based on views, votes, and activity';

SELECT
  COUNT(*) AS total_movies,
  MIN(release_year) AS earliest_year,
  MAX(release_year) AS latest_year
FROM
  workspace.default.movies_silver;